In [1]:
import base64
import pandas as pd
import os
import time
import openai
from PIL import Image
import io
import csv
from dotenv import load_dotenv

# Load from .env
load_dotenv()

# Access the key
openai_api_key = os.getenv("OPENAI_API_KEY")

# Setup OpenAI client (new in v1+)
client = openai.OpenAI(api_key=openai_api_key)


In [ ]:

prompt = """
    DO NOT SUMMARIZE. ONLY OUTPUT RAW TAGGED BLOCKS.
    Generate random QPE circuit with depth 8 and number of qubits 6 (Qiskit qasm), using logical gate, use only minimal classical register for measurement
For the circuit, do the following:

1. Start with a reasoning block in this format:
<thought>  
Based on the image, consist of pattern .. there is CCNOT and X gate as oracle then it is classified as Grover (as detail as possible) (Might be different for QPE, QML etc) 
(You may add more steps here as detail as possible)
(as detail as possible, you may add more here EXAMPLE: (not strictly following this)
- Based on the imagined image, the circuit starts with Hadamard gates on both qubits. This suggests a superposition state.
- The oracle appears to use X gates followed by CZ, then X gates again to mark a specific state like |11⟩ or |01⟩.
- The circuit uses 2 qubits: q[0], q[1].
- Classical registers are assumed: c[0], c[1].
- Total circuit depth is approximately 6:
   * Layer 1: Hadamard
   * Layer 2: X (pre-oracle)
   * Layer 3: CZ gate
   * Layer 4: Undo X
   * Layer 5: Diffusion
   * Layer 6: Measurement
- Oracle targets a specific basis state.
- No learned parameters or embeddings used.
- Logical gate layout is inferred visually.
)

2. Number of qubits, how many quantum classical registers, is there measurement, reverse engineering from image to thinking process, how to classify number of qubits, registers, and measurement

3. Composition - Logical GATE (reverse OCR):  
[Q0] --- [X] --- [H] --- [Cnot1]  
[Q1] --- [X] --- [H] --- [Cnot1]  
(as similiar and as detail as possible, dont use etc. or dont shorten this)
</thought>

2. Then output the full OpenQASM 2.0 code block in this format:
<OPENQASM code>
OPENQASM 2.0;
include "qelib1.inc";

qreg q[2];
creg c[2];

// Initialization
h q[0];
h q[1];

// Oracle for |11>
x q[0];
x q[1];
cz q[0], q[1];
x q[0];
x q[1];

// Diffusion
h q[0];
h q[1];
x q[0];
x q[1];
cz q[0], q[1];
x q[0];
x q[1];
h q[0];
h q[1];

// Measurement
measure q[0] -> c[0];
measure q[1] -> c[1];
</OPENQASM code>

Output only start with <think> .. </think> and end with <OPENQASM code> ... </OPENQASM code> sections for the circuit. Do not explain, summarize, or add commentary outside these tags.
"""


response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": "You are a quantum optics assistant."},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    # {
                    #     "type": "image_url",
                    #     "image_url": {
                    #         "url": f"data:image/png;base64,{base64_image}",
                    #         "detail": "high"
                    #     }
                    # }
                ]
            }
        ],
        max_tokens=2000
    )


In [3]:
print(response.choices[0].message.content)

<think>  
This quantum circuit is constructed for the Quantum Phase Estimation (QPE) algorithm, as indicated by the pattern of an initial series of Hadamard gates followed by controlled unitary operations with varying exponents, and an inverse Quantum Fourier Transform (QFT) towards the end. The circuit consists of 6 qubits in total, with the first 3 used as the counting register and the remaining 3 as the eigenstate register. There is only 1 classical register with 3 bits for measurement corresponding to the counting register outcomes.

Reverse engineering from the hypothetical image:
- The circuit starts by applying Hadamard gates to the counting register (q[0], q[1], q[2]), preparing a superposition across 3 qubits.
- A placeholder eigenstate is prepared on the last three qubits (q[3], q[4], q[5]) -- for variety, this state is left in |1⟩ using an X gate (commonly used for demonstration QPE circuits).
- Next, controlled unitary gates (here, for randomness and in line with logical ga

In [4]:
from qiskit import QuantumCircuit
from qiskit.visualization import circuit_drawer
import re

# Extract the QASM code block
match = re.search(r"<OPENQASM code>(.*?)</OPENQASM code>", response.choices[0].message.content, re.DOTALL)
qasm_code = match.group(1).strip() if match else None

 
# 1. Your OpenQASM code

# 2. Convert OpenQASM to QuantumCircuit
qc = QuantumCircuit.from_qasm_str(qasm_code)
 
# 3. Draw and save the circuit image
fig = circuit_drawer(qc, output='mpl')  # matplotlib figure
fig.savefig("circuit_output2.png")
 